<a href="https://colab.research.google.com/github/singleA4anchal/project/blob/main/Covid_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

In [3]:
pd.set_option("display.max_columns", 100)
sns.set(context="notebook", style="whitegrid", palette="viridis")

In [4]:
DATA_DIR = "../data"
COVID_PATH = os.path.join(DATA_DIR, "covid.csv")       # e.g., daily country-level COVID metrics
HAPPY_PATH = os.path.join(DATA_DIR, "happiness.csv")   # e.g., World Happiness Report-style file
OUT_DIR   = "../outputs"
FIG_DIR   = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

In [12]:

TARGET_YEARS = [2020, 2021]   # years to summarize COVID impact
AGG_METHOD   = "sum"          # "sum" or "mean" over the selected years
ROLLING_DAYS = 7              # smoothing window for time series plots

print("Plotly available:", PLOTLY_AVAILABLE)

Plotly available: True


In [10]:
# ============================================================
# 2) Basic Cleaning
# ============================================================
# - Standardize column names to snake_case.
# - Coerce dates and numeric types.
# - Drop obvious aggregates like "World" if present.
# - Keep country-level rows only.

def to_snake(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', s.strip()).lower()

# Minimal filtering
drop_countries = {'world','international','europe','asia','africa','oceania','north america','south america'}

# Coerce numeric for happiness metrics (adjust to your file)
num_cols_guess = ['happiness_score','gdp_per_capita','social_support',
                  'healthy_life_expectancy','freedom','generosity','perceptions_of_corruption']

NameError: name 'covid' is not defined

In [13]:
# ============================================================
# 3) Country Name Normalization (for joining)
# ============================================================
# Join keys often differ slightly across sources.
# Strategy:
# - Prefer an 'iso_code' if both datasets have it.
# - Else, standardize 'country' via a small mapping.

# Attempt to use ISO codes if available in both
has_iso_covid = 'iso_code' in covid.columns
has_iso_happy = 'iso_code' in happy.columns

if has_iso_covid and has_iso_happy:
    covid_key = 'iso_code'
    happy_key = 'iso_code'
else:
    covid_key = 'country'
    happy_key = 'country'
    # Light-touch harmonization mapping (extend as needed)
    name_map = {
        "united states": "united states",
        "usa": "united states",
        "uk": "united kingdom",
        "south korea": "korea, republic of",
        "russia": "russian federation",
        "iran": "iran, islamic republic of",
        "venezuela": "venezuela (bolivarian republic of)",
        "czechia": "czech republic",
        "taiwan": "taiwan*",
        "hong kong": "hong kong s.a.r. of china",
        # add dataset-specific fixes here after inspecting unmatched
    }
    covid['country_std'] = covid['country'].str.lower().map(lambda x: name_map.get(x, x)) if 'country' in covid.columns else None
    happy['country_std'] = happy['country'].str.lower().map(lambda x: name_map.get(x, x)) if 'country' in happy.columns else None
    covid_key = 'country_std'
    happy_key = 'country_std'

covid_keys_preview = covid[[k for k in [covid_key,'country'] if k in covid.columns]].drop_duplicates().head(10)
happy_keys_preview = happy[[k for k in [happy_key,'country'] if k in happy.columns]].drop_duplicates().head(10)
covid_keys_preview, happy_keys_preview


NameError: name 'covid' is not defined

In [14]:
# ============================================================
# 4) Exploratory Data Analysis (COVID)
# ============================================================
# - Basic missingness
# - Time series preview
# - Rolling averages for smoother plots

covid_info = covid.describe(include='all')
missing_covid = covid.isna().mean().sort_values(ascending=False).head(15)
covid_info, missing_covid


NameError: name 'covid' is not defined